# Milestone 2

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline,
)
from sentence_transformers import SentenceTransformer
from sentence_transformers import util as st_util

sys.path.append(os.path.abspath('..'))
from src.utils import map_at_3, load_config, set_seed
from src.preprocess import get_combined_texts

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

DEVICE     = 'mps' if torch.backends.mps.is_available() else 'cpu'
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH  = '../data/raw/train.csv'

df = pd.read_csv(TRAIN_PATH)
print(f'Device : {DEVICE}')
print(f'Shape  : {df.shape}')
print(f'iloc[0]: id={df.iloc[0]["id"]} | iloc[1]: id={df.iloc[1]["id"]}')

# Q1

In [2]:
hf_dataset = Dataset.from_pandas(df)

def make_combined_text(row):
    return {'combined_text': str(row['prompt']) + ' ' + str(row['A'])}

hf_dataset = hf_dataset.map(make_combined_text)

combined_at_51 = hf_dataset[51]['combined_text']
q1_answer = len(combined_at_51)

print(f'combined_text at index 51:\n{combined_at_51}')
print(f'Char Length: {q1_answer}')

Map: 100%|██████████| 2000/2000 [00:00<00:00, 31632.68 examples/s]

combined_text at index 51:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Char Length: 614


# Q2

In [3]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

q2_answer = tokenizer.vocab_size
print(f'Vocab Size: {q2_answer}')

Vocab Size: 30522


# Q3

In [4]:
q3_answer = tokenizer.sep_token_id

print(f'[SEP] token    : {tokenizer.sep_token}')
print(f'Q3 Result  : {q3_answer}')

[SEP] token    : [SEP]
Q3 Result  : 102


# Q4

In [5]:
all_prompts = df['prompt'].tolist()

encoded = tokenizer(
    all_prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

q4_answer = tuple(encoded['input_ids'].shape)
print(f'Input_ids shape: {q4_answer}')
print(f'{q4_answer[0]} rows × {q4_answer[1]} tokens')

Input_ids shape: (2000, 128)
2000 rows × 128 tokens


# Q5

In [6]:
hidden_size    = 768
num_heads      = 12
q5_answer      = hidden_size // num_heads

print(f'Hidden size  : {hidden_size}')
print(f'Num heads    : {num_heads}')
print(f'Dim: {q5_answer}')

Hidden size  : 768
Num heads    : 12
Dim: 64


# Q6

In [7]:
bert_model = AutoModel.from_pretrained('bert-base-uncased')
bert_model.eval()

row0_prompt = df.iloc[0]['prompt']
inputs_row0 = tokenizer(row0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs_row0 = bert_model(**inputs_row0)

last_hidden = outputs_row0.last_hidden_state
q6_answer   = tuple(last_hidden.shape)

print(f'Row index 0 prompt: {row0_prompt[:80]}...')
print(f'Hidden Shape: {q6_answer}')
print(f'batch={q6_answer[0]}, seq_len={q6_answer[1]}, hidden={q6_answer[2]}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25864.29it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Row index 0 prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Hidden Shape: (1, 31, 768)
batch=1, seq_len=31, hidden=768


# Q7

In [8]:
cls_vector    = last_hidden[0, 0, :]         
first_5       = cls_vector[:5].tolist()
q7_answer     = round(sum(first_5), 4)

print(f'[CLS] vector first 5 values: {[round(v, 6) for v in first_5]}')
print(f'Sum of first 5 values: {q7_answer}')

[CLS] vector first 5 values: [-0.467664, -0.075444, -0.201901, -0.007064, -0.448022]
Sum of first 5 values: -1.2001


# Q8

In [9]:
bert_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
bert_attn.eval()

target_text   = 'Light-ion fusion is a technique.'
inputs_attn   = tokenizer(target_text, return_tensors='pt')

# Find token index of 'fusion'
tokens        = tokenizer.convert_ids_to_tokens(inputs_attn['input_ids'][0])
print(f'Tokens: {tokens}')
fusion_index  = tokens.index('fusion')
print(f'fusion token index: {fusion_index}')

with torch.no_grad():
    outputs_attn = bert_attn(**inputs_attn)

# attentions: tuple of (num_layers,) each shape (batch, heads, seq, seq)
last_layer_attn = outputs_attn.attentions[-1]   # last layer
head0_attn      = last_layer_attn[0, 0, :, :]   # batch=0, head=0

# [CLS] is token index 0, attending to fusion
q8_answer = round(head0_attn[0, fusion_index].item(), 4)

print(f'Attention matrix shape (head 0): {head0_attn.shape}')
print(f'([CLS]→fusion weight): {q8_answer}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26870.12it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion token index: 4
Attention matrix shape (head 0): torch.Size([10, 10])
([CLS]→fusion weight): 0.1025


# Q9

In [11]:
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0_prompt  = df.iloc[0]['prompt']
row0_option_b = df.iloc[0]['B']

emb_prompt   = minilm.encode(row0_prompt,   convert_to_tensor=True)
emb_option_b = minilm.encode(row0_option_b, convert_to_tensor=True)

q9_answer = round(float(st_util.cos_sim(emb_prompt, emb_option_b)), 4)

print(f'Prompt    : {row0_prompt[:80]}...')
print(f'Option B  : {row0_option_b[:80]}...')
print(f'(cosine sim): {q9_answer}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7094.63it/s]


Prompt    : Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Option B  : Martin Heidegger believes that humans do not exist inside time, but that they ar...
(cosine sim): 0.7658


In [ ]:
# ── Q10: MAP@3 comparison — TF-IDF vs MiniLM pipeline ────────────────────────
ground_truth = df['answer'].tolist()

# Pipeline 1: TF-IDF (same as Milestone 1 Q10)
combined_texts = get_combined_texts(df)
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_vec.fit(combined_texts)

tfidf_preds = []
for _, row in df.iterrows():
    prompt_v = tfidf_vec.transform([str(row['prompt'])])
    sims = {col: float(cosine_similarity(prompt_v, tfidf_vec.transform([str(row[col])]))[0][0])
            for col in OPTION_COLS}
    tfidf_preds.append(sorted(sims, key=sims.get, reverse=True)[:3])

tfidf_map3 = round(map_at_3(tfidf_preds, ground_truth), 4)
print(f'TF-IDF pipeline MAP@3: {tfidf_map3}')

In [ ]:
# Pipeline 2: MiniLM embeddings
# Encode all prompts and options in batch for speed
print('Encoding prompts...')
prompt_embs = minilm.encode(df['prompt'].tolist(), convert_to_tensor=True, show_progress_bar=True)

option_embs = {}
for col in OPTION_COLS:
    print(f'Encoding option {col}...')
    option_embs[col] = minilm.encode(df[col].tolist(), convert_to_tensor=True, show_progress_bar=True)

minilm_preds = []
for i in range(len(df)):
    sims = {col: float(st_util.cos_sim(prompt_embs[i], option_embs[col][i]))
            for col in OPTION_COLS}
    minilm_preds.append(sorted(sims, key=sims.get, reverse=True)[:3])

minilm_map3 = round(map_at_3(minilm_preds, ground_truth), 4)
print(f'\nMiniLM pipeline MAP@3 : {minilm_map3}')
print(f'(MiniLM MAP@3): {minilm_map3}')

In [ ]:
# Count rows where TF-IDF misses but MiniLM hits
tfidf_miss_minilm_hit = 0
for i, truth in enumerate(ground_truth):
    tfidf_has  = truth in tfidf_preds[i]
    minilm_has = truth in minilm_preds[i]
    if not tfidf_has and minilm_has:
        tfidf_miss_minilm_hit += 1

print(f'(TF-IDF miss, MiniLM hit count): {tfidf_miss_minilm_hit}')

In [ ]:
# ── Q11: Zero-shot classification, row index 1 (2nd row), options A B C ───────
# This model is large (~1.6GB). Will download on first run.
zs_pipeline = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

row1        = df.iloc[1]                        # index 1 = 2nd row
row1_prompt = str(row1['prompt'])
candidates  = [str(row1['A']), str(row1['B']), str(row1['C'])]

zs_result   = zs_pipeline(row1_prompt, candidate_labels=candidates)

print(f'Prompt    : {row1_prompt[:80]}...')
print(f'Labels    : {zs_result["labels"]}')
print(f'Scores    : {[round(s, 4) for s in zs_result["scores"]]}')
print(f'Sum of 3 scores: {round(sum(zs_result["scores"]), 4)}')

q11_answer      = round(zs_result['scores'][0], 4)    # top-ranked probability
q11_scores_sum  = sum(zs_result['scores'])
print(f'(top-ranked probability): {q11_answer}')

In [ ]:
# ── Q12: Same but multi_label=True ────────────────────────────────────────────
zs_result_multi = zs_pipeline(row1_prompt, candidate_labels=candidates, multi_label=True)

print(f'Multi-label scores: {[round(s, 4) for s in zs_result_multi["scores"]]}')
print(f'Sum of 3 scores   : {round(sum(zs_result_multi["scores"]), 4)}')

q12_answer = round(abs(q11_scores_sum - sum(zs_result_multi['scores'])), 4)
print(f'\nSoftmax sum  : {round(q11_scores_sum, 4)}')
print(f'Sigmoid sum  : {round(sum(zs_result_multi["scores"]), 4)}')
print(f'(absolute difference): {q12_answer}')

In [ ]:
# ── Q13: Flan-T5 text generation, row index 0 ────────────────────────────────
flan_pipeline = pipeline('text2text-generation', model='google/flan-t5-small')

row0   = df.iloc[0]
prompt = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    f"Answer with just the letter A or B."
)

flan_output = flan_pipeline(prompt, max_new_tokens=5)
q13_answer  = flan_output[0]['generated_text']

print(f'Input prompt (first 120 chars): {prompt[:120]}...')
print(f'(exact output): "{q13_answer}"')